# Simulate Factory Scenarios
The following notebook uses simpy (an event simulator) which takes a factory scenario as an input, runs it with a set of factory workers with multiple skill levels to calculate total rework, scrap and products produced by each station and the whole factory

In [8]:
import simpy
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import trange, tqdm
import os
import math
# import logging

The function below simulates productivity loss over a given shift

In [9]:
def time_impact(x, max_x):
    t = x%max_x
    if t <= 0:
        return 0
    elif t >= max_x:
        return 1
    else:
        k = 12 / max_x  # Steepness factor
        x0 = max_x / 2  # Midpoint
        return 1 / (1 + math.exp(-k * (t - x0)))
    
shift_time = 12

The classes below define the worker and the process

In [10]:
# Set up logging
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s', datefmt='%H:%M:%S')

class Worker:
    def __init__(self, skill):
        self.skill = skill
        self.is_busy = False

class Process:
    def __init__(self, env, process_data, initial_data, workers, process_dict):
        self.env = env
        self.name = process_data['name']
        self.process_id = process_data['process_id']
        self.wip_output_to = process_data['wip_output_to']
        self.inventory_input_items = initial_data['inventory_input_items']
        self.WIP_input_items = initial_data['WIP_input_items']
        self.output_item = process_data['output_item']
        self.base_processing_time = process_data['base_processing_time']
        self.complexity = process_data['complexity']
        self.processing_capacity = process_data['processing_capacity']
        self.workers = workers
        
        self.required_inventory = process_data['inventory_input_items']
        self.required_wip = process_data['WIP_input_items']
        
        self.outputs = 0
        self.scrap = 0
        self.rework = 0
        self.total_run_time = 0
        self.is_paused = True
        self.process_dict = process_dict
        self.start_time = None
        self.end_time = None
        
    def run(self):
        while True:
            # Check if we have enough inputs and a free worker
            if (all(self.inventory_input_items[item] >= self.required_inventory[item] for item in self.required_inventory) and
                all(self.WIP_input_items[item] >= self.required_wip[item] for item in self.required_wip)):
                
                # Find an available worker
                available_workers = [w for w in self.workers if not w.is_busy]
                if available_workers:
                    worker = random.choice(available_workers)
                    worker.is_busy = True
                    
                    if self.is_paused:
                        self.is_paused = False
                        if self.start_time is None:
                            self.start_time = self.env.now
                        # logging.info(f"Time {self.env.now:.2f}: Process {self.process_id} unpaused")
                    
                    # Consume inputs
                    for item, qty in self.required_inventory.items():
                        self.inventory_input_items[item] -= qty
                    for item, qty in self.required_wip.items():
                        self.WIP_input_items[item] -= qty
                    
                    # Process
                    start_time = self.env.now
                    processing_time = self.base_processing_time / worker.skill * (1 + 0.3*time_impact(self.env.now, shift_time)) if self.name in ['assembly', 'electronics'] else self.base_processing_time / worker.skill
                    yield self.env.timeout(processing_time)
                    self.total_run_time += self.env.now - start_time
                    
                    # Handle scrap and rework
                    if random.random() < self.complexity * (1 - worker.skill):
                        if random.random() < 0.5:  # 50% chance of scrap vs rework
                            self.scrap += 1
                            # logging.info(f"Time {self.env.now:.2f}: Process {self.process_id} produced scrap")
                        else:
                            self.rework += 1
                            # logging.info(f"Time {self.env.now:.2f}: Process {self.process_id} started rework")
                            yield self.env.timeout(processing_time * 0.3)  # 30% more time for rework
                            self.total_run_time += processing_time * 0.3
                            # logging.info(f"Time {self.env.now:.2f}: Process {self.process_id} completed rework")
                    else:
                        self.outputs += 1
                        # logging.info(f"Time {self.env.now:.2f}: Process {self.process_id} produced output")
                        
                        # Send output to next process
                        if self.wip_output_to:
                            next_process = self.process_dict.get(self.wip_output_to)
                            if next_process:
                                next_process.WIP_input_items[self.output_item] += 1
                                # logging.info(f"Time {self.env.now:.2f}: Process {self.process_id} sent output to {self.wip_output_to}")
                    
                    # Free the worker
                    worker.is_busy = False
                else:
                    yield self.env.timeout(1)  # Wait for a worker to become available
            else:
                if not self.is_paused:
                    self.is_paused = True
                    self.end_time = self.env.now
                    # logging.info(f"Time {self.env.now:.2f}: Process {self.process_id} paused")
                yield self.env.timeout(1)  # Wait and check again

The functions below run the factory and processes

In [11]:
def load_json(filename):
    with open(filename, 'r') as f:
        return json.load(f)
    
# Run simulation respecting dependencies
def run_factory(env, factory):
    processes = [env.process(p.run()) for group in factory for p in group]
    yield env.all_of(processes)

def getVals (scenario, workers_data, initial_conditions, size):
    # Set up environment
    env = simpy.Environment()

    # Create workers
    workers = [Worker(w['skill']) for w in workers_data['workers']]

    lenWorker = len(workers)

    if lenWorker < 5:
        runtime = 20000 if size >16 else 10000
    else:
        runtime = 5000
    # print('small' if runtime == 10000 else 'large')

    # Create processes respecting dependencies
    factory = []
    process_dict = {}

    for process_group in scenario['assembly_sequence']:
        group_processes = []
        for process_data in process_group:
            initial_process_data = next(p for p in initial_conditions['assembly_sequence'][scenario['assembly_sequence'].index(process_group)] if p['process_id'] == process_data['process_id'])
            process = Process(env, process_data, initial_process_data, workers, process_dict)
            group_processes.append(process)
            process_dict[process.process_id] = process
        factory.append(group_processes)

    # Run simulation
    env.process(run_factory(env, factory))
    env.run(until=runtime)  # Set a time limit to prevent infinite loops

    # Calculate actual factory runtime
    all_processes = [p for group in factory for p in group]
    factory_start_time = min(p.start_time for p in all_processes if p.start_time is not None)
    factory_end_time = max(p.end_time for p in all_processes if p.end_time is not None)
    total_factory_run_time = factory_end_time - factory_start_time

    # Generate report
    total_factory_output = process_dict[scenario['assembly_sequence'][-1][0]['process_id']].outputs

    return scenario, workers_data, process_dict, total_factory_output, total_factory_run_time

The function below generates the report

In [12]:
def generate_report(scenario, workers_data, process_dict, total_factory_output, total_factory_run_time):
    def create_process_tree(process_id):
        process = process_dict[process_id]
        
        # Calculate total consumed WIPs and Inventory items
        consumed_wips = {item: process.required_wip[item] * process.outputs for item in process.required_wip}
        consumed_inventory = {item: process.required_inventory[item] * process.outputs for item in process.required_inventory}
        
        features = list(consumed_wips.values()) + list(consumed_inventory.values())
        feature_names = list(consumed_wips.keys()) + list(consumed_inventory.keys())
        
        output = {
            "Total_WIPs_produced": process.outputs,
            "total_rework_produced": process.rework,
            "total_scrap_produced": process.scrap,
            "station_Active_time": process.total_run_time,
            "throughput": process.outputs / process.total_run_time if process.total_run_time > 0 else 0
        }
        
        node = {
            "module_id": process.process_id,
            "module_name": process.name,
            "features": features,
            "feature_names": feature_names,
            "output": output,
            "total_output": None,
            "left_child_name": None,
            "right_child_name": None,
            "children": []
        }
        
        # Find predecessor processes
        predecessors = [p for p in process_dict.values() if p.wip_output_to == process_id]
        for predecessor in predecessors:
            child_tree = create_process_tree(predecessor.process_id)
            node["children"].append(child_tree)
        
        return node

    # Find the last process in the assembly sequence
    last_process_id = scenario['assembly_sequence'][-1][0]['process_id']
    
    json_tree = create_process_tree(last_process_id)
    
    report = {
        "module_params_dict": None,
        "query_id": scenario['scenario_id'],
        "treatment_id": workers_data['distribution'],
        "query_output": {
            "total_output": total_factory_output,
            "factory_run_time": total_factory_run_time,
            "json_tree": json_tree
        }
    }
    
    return report

The code block below runs the factory and saves the reports 

In [14]:
scenarios = list(range(1, 51))
workers = ['00', '01']
demands = list(range(5, 1005, 5))

top_folder = 'simulation_report_normal'
os.makedirs(top_folder, exist_ok=True)

for worker in workers:
    workers_data = load_json(f'workers/workers_{worker}.json')
    worker_folder = f'{top_folder}/worker_{worker}'
    os.makedirs(worker_folder, exist_ok=True)
    for scenario_num in tqdm(scenarios):
        scenario = load_json(f'factory_scenario/scenario_{scenario_num}.json')
        file_size_in_kb = os.path.getsize(f'factory_scenario/scenario_{scenario_num}.json')/1024
        scenario_folder = f'{worker_folder}/scenario_{scenario_num}'
        os.makedirs(scenario_folder, exist_ok=True)
        for demand in demands:
            for trial in range(10):
                initial_conditions = load_json(f'initial_conditions/scenario_{scenario_num}/scenario_{scenario_num}_demand_{demand}.json')
                scenario_P, workers_data_P, process_dict_P, total_factory_output_P, total_factory_run_time_P = getVals(scenario, workers_data, initial_conditions, file_size_in_kb)
                report = generate_report(scenario_P, workers_data_P, process_dict_P, total_factory_output_P, total_factory_run_time_P)
                report_name = f'{scenario_folder}/scenario_{scenario_num}_demand_{demand}_trial_{trial}_report.json'
                with open(report_name, 'w') as f:
                    json.dump(report, f, indent=2)

  0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 